# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakhawat-4/my-ml-internship-v2/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: RANKING
Why: Order pages by importance for review
Action: Top pages get reviewed first

In [1]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
# Ranking demo: order pages by a review-worthiness proxy, don't threshold them into classes
demo_ranked = df.assign(content_score=df["impressions_90d"] * (1 - df["ctr"])).sort_values(
    "content_score", ascending=False)[["content_id", "impressions_90d", "ctr", "content_score"]]
print("Top 3 by rank:")
print(demo_ranked.head(3).to_string(index=False))
print("\nBottom 3 by rank:")
print(demo_ranked.tail(3).to_string(index=False))

Top 3 by rank:
          content_id  impressions_90d  ctr  content_score
content_2cb567c3c89b           497727 0.10       447954.3
content_5fe46e04994d           517715 0.14       445234.9
content_8c19996aa890           509252 0.15       432864.2

Bottom 3 by rank:
          content_id  impressions_90d  ctr  content_score
content_a22b7f6c73c5            28192 4.78     -106565.76
content_c8ad1f4d0e56            62927 3.40     -151024.80
content_07e0b9af8b1a           214816 1.94     -201927.04


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_high_value (1=valuable, 0=not)
Proxy: content_score > median

In [2]:
df["content_score"] = df["impressions_90d"] * (1 - df["ctr"])
df["is_high_value"] = (df["content_score"] > df["content_score"].median()).astype(int)
print(df["is_high_value"].value_counts())
print(f"\nProxy label is a MEDIAN SPLIT on an engineered score, not an observed outcome —"
      f" it is a defined rule, so it must be reported as directional, not causal.")

is_high_value
0    15002
1    14998
Name: count, dtype: int64

Proxy label is a MEDIAN SPLIT on an engineered score, not an observed outcome — it is a defined rule, so it must be reported as directional, not causal.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: Precision@50
Target: > 70%
Meaning: % of top 50 recommendations that are valuable

In [3]:
def precision_at_k(y_true, scores, k):
    ranked = pd.Series(scores).sort_values(ascending=False).index[:k]
    return y_true.iloc[ranked].mean()

# Demo with the proxy label above, ranking by content_score itself (sanity check only)
demo_p50 = precision_at_k(df["is_high_value"], df["content_score"], 50)
print(f"Demo Precision@50 (ranking by the same score the label was cut from): {demo_p50:.3f}")
print("This demo is circular by construction (label derived from the same score) —")
print("it only confirms the metric function works; the real model must be judged on")
print("held-out data with a metric that does not share the label's own formula.")

Demo Precision@50 (ranking by the same score the label was cut from): 1.000
This demo is circular by construction (label derived from the same score) —
it only confirms the metric function works; the real model must be judged on
held-out data with a metric that does not share the label's own formula.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("One row = one page")
display(df.head())
df["is_high_value"] = df["content_score"] if "content_score" in df.columns else (df["impressions_90d"] * (1 - df["ctr"])) > (df["impressions_90d"] * (1 - df["ctr"])).median()
display(df[["content_id"]].join(df.filter(like="impressions_90d")).head())

One row = one page


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


,content_id,impressions_90d
0,content_304f48230142,3803
1,content_a1fb4e703a9e,15320
2,content_9aa793d4d895,12581
3,content_331d6c4de07b,11751
4,content_d99b7a2d90ca,19140


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML is better:
1. Uses ALL 44 features
2. Finds hidden patterns
3. Learns from new data
4. Optimizes for Precision@50
5. Rules are static and fail

In [5]:
# Why a fixed if/else rule struggles: high-impression pages exist at both
# high AND low CTR, so a single threshold on one field misclassifies many pages.
overlap = df.groupby(pd.qcut(df["impressions_90d"], 4, duplicates="drop"))["ctr"].agg(["mean", "std"])
print(overlap)
print("\nCTR varies a lot *within* every impression band (std is a sizeable fraction of the")
print("mean in each row) -- a single-field cutoff rule can't capture that, which is the")
print("concrete case for a model that can weigh several fields together.")

                         mean       std
impressions_90d                        
(0.999, 81.0]        1.265650  6.459379
(81.0, 731.0]        0.237681  0.561408
(731.0, 3615.25]     0.228640  0.307763
(3615.25, 517715.0]  0.310549  0.318456

CTR varies a lot *within* every impression band (std is a sizeable fraction of the
mean in each row) -- a single-field cutoff rule can't capture that, which is the
concrete case for a model that can weigh several fields together.


## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.